In [6]:
# Imports & Config

from medical_tokenizer import *
import os
import pickle
import json

INPUT_PATH = "pubmed_filtered_corpus.txt"
VOCAB_SIZE = 32_000
SPECIAL_TOKENS = MEDICAL_SPECIAL_TOKENS
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# Download Dataset & Generate Corpus

# Option A: Download fresh from HuggingFace (run once, then comment out)
from datasets import load_dataset
ds = load_dataset("ccdv/pubmed-summarization", split="train[:10%]")
create_filtered_medical_corpus(ds, INPUT_PATH)

# Option B: If you already have the corpus file, skip this cell entirely.
# Just make sure INPUT_PATH points to your existing file, e.g.:
# INPUT_PATH = "pubmed_filtered_corpus.txt"

In [ ]:
# Train the Tokenizer
vocab, merges = train_medical_bpe_tokenizer(
    input_path=INPUT_PATH,
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    use_medical_regex=True,
)

In [ ]:
# Save Results

vocab_json = {str(token_id): vocab[token_id].decode("utf-8", errors="replace") 
                for token_id in vocab}

# Create reverse mapping: bytes → token_id
bytes_to_id = {v: k for k, v in vocab.items()}

# Convert merges: tuple of bytes → list of strings
merges_json = []
for pair in merges:
    left_id = bytes_to_id.get(pair[0])
    right_id = bytes_to_id.get(pair[1])
    if left_id is not None and right_id is not None:
        merges_json.append([vocab[left_id].decode("utf-8", errors="replace"), vocab[right_id].decode("utf-8", errors="replace")])

# Save as JSON
# with open(f"{RESULTS_DIR}/vocab.json", "w", encoding="utf-8") as f:
#     json.dump(vocab_json, f, indent=2, ensure_ascii=False)

# with open(f"{RESULTS_DIR}/merges.json", "w", encoding="utf-8") as f:
#     json.dump(merges_json, f, indent=2, ensure_ascii=False)

# Save as pickle 
with open(f"{RESULTS_DIR}/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)
with open(f"{RESULTS_DIR}/merges.pkl", "wb") as f:
    pickle.dump(merges, f)


In [8]:
# Load Tokenizer
tokenizer = MedicalBPETokenizer(RESULTS_DIR)

Loaded : 32,000 tokens | 31,743 merges


In [9]:
# Test

sent = "Patient treated with IL-6 inhibitors for rheumatoid arthritis."
ids = tokenizer.encode(sent)
decoded = tokenizer.decode(ids)
print(f"Input:   '{sent}'")
print(f"Tokens:  {ids}")
print(f"Decoded: '{decoded}'")

Input:   'Patient treated with IL-6 inhibitors for rheumatoid arthritis.'
Tokens:  [80, 349, 295, 1391, 325, 32, 73, 76, 45, 54, 3480, 331, 9763, 6578, 46]
Decoded: 'Patient treated with IL-6 inhibitors for rheumatoid arthritis.'
